# N081 · BFS、最短步数与多源扩散

**目标：** 在等权边图上按距离层发现节点。

**先修：** N053, N079, N080。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 队列；入队即标记；多源初始化；网格最短路；不可达。

**配套讲解来源：** [同名逐章意见](../../comment/081_graph_bfs_multisource.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

上一章 DFS 回答"能不能到"，这一章 BFS 回答"最少几步能到"——前提是每条边的代价都一样（都是 1 步）。BFS 像往水里扔石头激起的波纹：从起点一圈一圈往外扩，第一次碰到谁，谁的距离就定下来了。

本章三个具体问题：

1. **多源最短距离：** 图里有一组起点，求每个点到"最近的一个起点"的距离。具体例子：邻接表 `adj=[[1],[0,2],[1,3],[2],[]]`（0—1—2—3 一条链，4 号孤立），起点集合 `[0,3]`，输出 `[0,1,1,0,-1]`——0 和 3 自己距离 0；1 离 0 最近（1 步）；2 离 3 最近（1 步）；4 谁也够不着，是 -1。
2. **腐烂的橘子：** 网格里 0=空、1=新鲜橘子、2=烂橘子。每分钟烂橘子把上下左右相邻的新鲜橘子传染烂掉，问几分钟全部烂完；有橘子永远烂不到就返回 -1。具体例子：`[[2,1,1],[1,1,0],[0,1,1]]` 输出 4——烂从左上角开始扩散，最右下角那颗要 4 分钟才轮到。
3. **01 矩阵：** 每个格子里是 0 或 1，求每个 1 格子到最近 0 格子的距离（走上下左右）。具体例子：`[[0,1],[1,1]]` 输出 `[[0,1],[1,2]]`——(0,1) 旁边就是 0 距离 1；(1,1) 到 (1,0) 或 (0,1) 都要 2 步。

这三题的共同点：**所有"源头"（起点、烂橘子、0 格子）从第 0 层同时开始扩散**——这就是多源 BFS。

## 2. 基础知识：先读懂这些词

- **BFS（广度优先搜索）：** 用队列实现的逐层扩张：先访问离起点 1 步的所有点，再访问 2 步的，依此类推。因为队列先进先出，先入队的距离必然不比后入队的大，这个"从近到远"的顺序就是它能算最短路的根本原因。
- **等权边（单位边权）：** 每走一步代价相同。BFS 只在这种情况下保证最短路；一旦边权有 0 有 5（下一章 N084 的主题），普通队列的顺序就不够用了。
- **队列（deque）：** 双端队列。`popleft()` 从左端出队（近的先走），`append()` 从右端入队（新的排后面），两端操作都是 O(1)。用普通 list 的 `pop(0)` 出队是 O(n)，大图上会慢一个量级。
- **入队即标记：** 一个点**放进队列的那一刻**就把距离写上，而不是等它出队才标记。若出队才标，同一个点可能在队列里积压好几份，浪费还容易算错。
- **多源 BFS：** 把所有源点在开始时统统塞进队列、距离全设 0，然后照常跑 BFS。效果等价于"每个点到离它最近的那个源的距离"。它不是"从随便某一个源出发"，而是所有源并列在第 0 层。
- **网格的隐式图与四方向：** 格子是点，上下左右相邻是边，**不含对角线**。方向表 `[(r+1,c),(r-1,c),(r,c+1),(r,c-1)]` 加边界检查就是全部图结构。
- **不可达（-1 约定）：** BFS 跑完队列空了，某些点的距离还是初始的 -1，说明没有路径能到它。代码约定用 -1 表示"够不着"，而不是留个假的有限数字。
- **距离数组当 visited 用：** 本章所有实现都不另开 visited 集合，而是用"距离是否还是 -1"来判断访问过没有——一个数组干两份活。

## 3. 思路推导：从小例子开始

**Step 1：初始化第 0 层。** 把所有源点入队、距离设 0。单源就是一个源；多源就是全部源一起上。

**Step 2：循环出队、逐层扩散。** 每次从队首拿出一个点 u，扫描它的邻居 v：如果 v 还没距离（-1），就设为 distance[u]+1 并入队。

**Step 3：为什么第一次碰到就是最短：** 队列里的距离从头到尾只会相等或递增（最多差 1）。轮到 u 出队时，比 u 近的层早就处理完了，所以 u 的距离不可能再被改善；同理第一次给 v 定的距离就是最终答案。

**Step 4：按题意包装。** 橘子题把"距离"读成"时间"，最后检查有没有新鲜橘子幸存；01 矩阵题把 0 格子当源，直接输出距离矩阵。

**手算演示（update_matrix([[0,1],[1,1]])）：**

```
初始:  d=[[0,-1],     队列: [(0,0)]
       [-1,-1]]
```

- 源是所有 0 格子：只有 (0,0)，d[0][0]=0，入队。
- 出队 (0,0)。邻居：(1,0) 是 -1 → d[1][0]=0+1=1，入队；(0,1) 是 -1 → d[0][1]=1，入队。
- 出队 (1,0)。邻居：(0,0) 已有距离，跳过；(1,1) 是 -1 → d[1][1]=1+1=2，入队。
- 出队 (0,1)。邻居 (1,1) 已有距离 2，跳过（这步很重要：先到先得，不覆盖）。
- 出队 (1,1)，邻居都有距离。队列空，返回 `[[0,1],[1,2]]`。

**手算演示（oranges_rotting([[2,1,1],[1,1,0],[0,1,1]]) 的层数）：** 把烂橘子 (0,0) 当第 0 分钟层。第 1 分钟传染 (0,1) 和 (1,0)；第 2 分钟传染 (0,2) 和 (1,1)；第 3 分钟传染 (2,1)；第 4 分钟传染 (2,2)。fresh 从 7 数到 0，最后出队的 time=4，返回 4。你可以自己把这 4 圈画在纸上，正好是波纹一层层推过去。

**手算演示（bfs_distances 链上多源）：** 起点 [0,3] 全部距离 0、一起入队。出队 0 → 1 得距离 1；出队 3 → 2 得距离 1；出队 1、2 后无新点。4 号点从头到尾没进队列，保持 -1。注意 2 号离 0 其实是 2 步、离 3 是 1 步，答案取 1——这就是"多源求最近"的含义。

## 4. 核心代码：bfs_distances

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def bfs_distances(adj, sources):
    distance = [-1] * len(adj)
    queue = deque()
    for s in sources:
        if distance[s] == -1:
            distance[s] = 0
            queue.append(s)
    while queue:
        u = queue.popleft()
        for v in adj[u]:
            if distance[v] == -1:
                distance[v] = distance[u] + 1
                queue.append(v)
    return distance
```

### 逐段读懂代码

**第一段：bfs_distances——最朴素的多源 BFS 模板。**

```python
from collections import deque

def bfs_distances(adj, sources):
    distance = [-1] * len(adj)
    queue = deque()
    for s in sources:
        if distance[s] == -1:
            distance[s] = 0
            queue.append(s)
    while queue:
        u = queue.popleft()
        for v in adj[u]:
            if distance[v] == -1:
                distance[v] = distance[u] + 1
                queue.append(v)
    return distance
```

- `distance=[-1]*len(adj)`：初始全 -1，一物两用——既表示"未访问"，也直接当返回值（够不着的点保持 -1）。
- 初始化循环里 `if distance[s]==-1`：挡**重复源**。sources 里同一个点出现两次，第二次直接跳过，不然队列里会有两份。
- 主体两行核心：出队 u；对每个还没距离的邻居 v，写入 `distance[u]+1` 并入队。**写入和入队同时发生**，这就是"入队即标记"。
- 没有单独的 visited 集合，`distance[v]==-1` 就是访问判断。

**第二段：oranges_rotting——多源扩散 + 幸存者检查。**

（对应程序见下方分段实现与完整代码。）

- `if not grid or not grid[0]: return 0`：空网格视为 0 分钟（没有橘子需要烂）。
- `a=[row[:] for row in grid]`：逐行浅拷贝一份再改。烂橘子要把格子从 1 改成 2，不拷贝就会弄脏调用者的原数组——和上一章 N080 用 set 不同，这里直接在网格上打标记（改成 2 就天然"不会再次入队"）。
- 第一遍双循环收集所有源（2 格子，携带起始时间 0）并数出新鲜橘子总数 fresh。
- 队列元素是三元组 `(行,列,当前时间)`。出队时 `r,c,time=queue.popleft()` 一行解包三个值，time 就是这颗橘子被传染的分钟数。
- 扩散条件三连：界内、是新鲜橘子（a==1）、（改成 2 后自动不会重复处理）。传染动作三连：改 2、fresh 减 1、带 time+1 入队。
- `return -1 if fresh else time`：队列跑完还有 fresh>0，说明有橘子被隔开永远烂不到，按题意返回 -1；否则 time 恰好是**最后一次**出队的值——因为队列时间单调不降，最后出队的就是最晚烂掉的。注意 fresh=0 且队列为空（全是空格/烂橘）时返回初始 time=0，正确。

**第三段：update_matrix——所有 0 当源的反向扩散。**

```python
def update_matrix(mat):
    if not mat or not mat[0]:
        return []
    rows, cols = (len(mat), len(mat[0]))
    d = [[-1] * cols for _ in range(rows)]
    queue = deque()
    for r in range(rows):
        for c in range(cols):
            if mat[r][c] == 0:
                d[r][c] = 0
                queue.append((r, c))
    while queue:
        r, c = queue.popleft()
        for nr, nc in [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]:
            if 0 <= nr < rows and 0 <= nc < cols and (d[nr][nc] == -1):
                d[nr][nc] = d[r][c] + 1
                queue.append((nr, nc))
    return d
```

- 关键在第一遍循环的方向：**不是从每个 1 出发找 0，而是把所有 0 当源反向扩散**。如果对每个 1 格子单独跑一次 BFS，最坏 O((RC)²)；反过来所有 0 并列第 0 层、一次 BFS 全图覆盖，只有 O(RC)。这是"多源换视角"最经典的省时技巧。
- `d=[[-1]*cols for _ in range(rows)]`：注意写法，不能用 `[[-1]*cols]*rows`——那会是同一行列表的重复引用（N079 讲过同一个坑）。
- 主体和 oranges_rotting 同构：距离 -1 当未访问标记，四方向加边界，距离加 1 入队。
- 值得一提：notebook 的 “正确性与复杂度”部分 说"全 1 无 0 的矩阵是教学扩展，返回 -1 而非伪造距离"——没有 0 时队列初始为空，d 保持全 -1，函数诚实返回全 -1。

## 5. 分段实现：按顺序运行

**本章接口：** `bfs_distances(adj, sources)`、`oranges_rotting(grid)`、`update_matrix(mat)`

### 导入本章使用的库

In [1]:
from collections import deque

### bfs_distances

In [2]:
def bfs_distances(adj, sources):
    distance = [-1] * len(adj)
    queue = deque()
    for s in sources:
        if distance[s] == -1:
            distance[s] = 0
            queue.append(s)
    while queue:
        u = queue.popleft()
        for v in adj[u]:
            if distance[v] == -1:
                distance[v] = distance[u] + 1
                queue.append(v)
    return distance

### oranges_rotting

In [3]:
def oranges_rotting(grid):
    if not grid or not grid[0]:
        return 0
    a = [row[:] for row in grid]
    rows, cols = (len(a), len(a[0]))
    queue = deque()
    fresh = 0
    for r in range(rows):
        for c in range(cols):
            if a[r][c] == 2:
                queue.append((r, c, 0))
            if a[r][c] == 1:
                fresh += 1
    time = 0
    while queue:
        r, c, time = queue.popleft()
        for nr, nc in [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]:
            if 0 <= nr < rows and 0 <= nc < cols and (a[nr][nc] == 1):
                a[nr][nc] = 2
                fresh -= 1
                queue.append((nr, nc, time + 1))
    return -1 if fresh else time

### update_matrix

In [4]:
def update_matrix(mat):
    if not mat or not mat[0]:
        return []
    rows, cols = (len(mat), len(mat[0]))
    d = [[-1] * cols for _ in range(rows)]
    queue = deque()
    for r in range(rows):
        for c in range(cols):
            if mat[r][c] == 0:
                d[r][c] = 0
                queue.append((r, c))
    while queue:
        r, c = queue.popleft()
        for nr, nc in [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]:
            if 0 <= nr < rows and 0 <= nc < cols and (d[nr][nc] == -1):
                d[nr][nc] = d[r][c] + 1
                queue.append((nr, nc))
    return d

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

mat=[[0,0,0],[0,1,0],[1,1,1]]; distance=update_matrix(mat)
show_table(['行','原网格','最近0距离'],[(i,mat[i],distance[i]) for i in range(len(mat))])

行,原网格,最近0距离
0,"[0, 0, 0]","[0, 0, 0]"
1,"[0, 1, 0]","[0, 1, 0]"
2,"[1, 1, 1]","[1, 2, 1]"


## 7. 正确性、适用条件与复杂度

队列距离非递减。若某顶点首次发现距离d却有更短路径，则该路径前驱应更早出队并提前发现它，矛盾。多源只是改变初始层，不改变证明。

**代价：** 图O(V+E)时间/O(V)空间；网格O(RC)。全1无0的矩阵是教学扩展，返回−1而非伪造有限距离。

### 展开理解

**为什么第一次给的距离就是最短：** 我们用"队列里的距离不回头"这一条性质来论证。入队顺序保证：任何时刻，队列里前面的元素距离 ≤ 后面的元素距离（新入队的距离最多比刚出队的大 1）。假设某个点 v 第一次被定为距离 d，但其实存在更短的路径 d' < d。那这条更短路径上，v 的前一跳 u' 距离是 d'-1 < d-1，u' 会比"给 v 定距离的那个 u"更早出队；u' 出队时扫邻居就会发现 v 还没距离，当场把 v 定成 d'。矛盾。所以第一次的值不可能被更短路径推翻。

**多源为什么不用重新证明：** 多源只是把第 0 层从"一个点"换成"一排点"，队列"从头到尾距离不降"的性质原封不动，上面的论证照搬即可。

**为什么入队即标记：** 如果等出队才标记，一个点可能在队列里被塞进好几份（多个邻居都发现它），虽然结果往往仍正确，但白白多处理；更糟的是在带权或写法不严时容易出错。入队即标记保证每点恰好入队一次。

**复杂度（用具体数字感受）：** 图版 BFS 时间 O(V+E)、空间 O(V)：10 万点 20 万边约 30 万次操作，瞬间完成。网格版 O(RC)：300×400 的橘子网格是 12 万格，每格常数次方向检查，同样毫秒级。对比反例：01 矩阵若对每个 1 单独 BFS，300×400 全 1 网格要 12 万 × 12 万 = 百亿级操作，跑不完——这就是反向多源的价值。

## 8. 单元测试：每个用例在检查什么

```python
assert oranges_rotting([[1, 1]]) == -1
```
**特殊值**：两颗新鲜橘子、零颗烂橘子。没有源头，永远烂不完，必须返回 -1（这条抓"忘了幸存者检查"的 bug）。

```python
assert oranges_rotting([[0, 2]]) == 0
```
**边界值**：有烂橘子但没有新鲜橘子。第 0 分钟就"全部烂完"，答案是 0 而不是 -1，测 fresh==0 分支和 time 初始值。

```python
assert oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]) == 4
```
正常值：第三节手算过的例子，最远橘子 4 分钟烂掉，同时验证扩散按分钟逐层推进、不是瞬间全烂。

```python
assert update_matrix([[0, 1], [1, 1]]) == [[0, 1], [1, 2]]
```
正常值：第四节逐步演过的 2×2 例子。右下角距离 2 而不是 1，测对角线不算相邻（否则会是 1）。

```python
assert update_matrix([[1]]) == [[-1]]
```
**教学扩展边界**：全 1、没有任何 0。函数约定返回 -1 表示够不着，测"没有源时队列空转、诚实输出 -1"。

```python
adj = [[1], [0, 2], [1, 3], [2], []]

assert bfs_distances(adj, [0, 3]) == [0, 1, 1, 0, -1]
```
正常值 + 不可达：两个源并列第 0 层；点 2 离源 3 是 1 步（虽然离源 0 是 2 步），测"取最近源"；点 4 是孤立点，保持 -1，测不可达约定。

```python
for _ in range(60): ... assert bfs_distances(adj,sources)==ref
```
**随机对拍**：60 组随机 7 点图、固定三个源 [0,3,5]。参考答案 ref 的算法是：对每个源单独跑单源 BFS，每个点取"所有可达源距离的最小值"。多源一次跑 == 逐源跑再取 min，两组结果必须逐点相等——这正是"多源求最近源"的定义式验证。

In [6]:
assert oranges_rotting([[1, 1]]) == -1

assert oranges_rotting([[0, 2]]) == 0

assert oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]) == 4

assert update_matrix([[0, 1], [1, 1]]) == [[0, 1], [1, 2]]

assert update_matrix([[1]]) == [[-1]]

adj = [[1], [0, 2], [1, 3], [2], []]

assert bfs_distances(adj, [0, 3]) == [0, 1, 1, 0, -1]

from random import Random

rng = Random(81)

for _ in range(60):
    n = 7
    adj = [[j for j in range(n) if i != j and rng.randrange(4) == 0] for i in range(n)]
    sources = [0, 3, 5]
    singles = [bfs_distances(adj, [s]) for s in sources]
    ref = [min([d[v] for d in singles if d[v] >= 0], default=-1) for v in range(n)]
    assert bfs_distances(adj, sources) == ref

print('N081: 所有本章断言通过')

N081: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 把单源搜索改成多源。

**练习 2：** 解释一般非等权边不能直接用普通BFS。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（单源改多源）：** 思路方向：拿一个只会单源 BFS 的版本（比如从唯一源出发），改造初始化那一小段。提示：单源版是 `distance[s]=0; queue=[s]`，多源版只是把它套进 for 循环并对重复源去重——主体 while 循环一行都不用改。手算示例可以用本章的链图 adj=[[1],[0,2],[1,3],[2],[]]，先预测 sources=[0] 和 [0,3] 两种输出的差别，再运行对照。

**练习 2（非等权边为什么不能普通 BFS）：** 思路方向：构造一个反例就够了。提示：图 0→1 边权 1，0→2 边权 1，2→1 边权 0。普通 BFS 从 0 出发会先碰 1（距离 1）并封死，但真实最短路是 0→2→1 总共 1。问题出在"队列距离不降"被 0 权边破坏：后入队的可以和前面一样近。手算这个 3 点例子在普通 BFS 和正确答案下的两张距离表，差异一目了然；顺带想想它正是下一章 0-1 BFS（N084）要修的 bug。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
from collections import deque

def bfs_distances(adj, sources):
    distance = [-1] * len(adj)
    queue = deque()
    for s in sources:
        if distance[s] == -1:
            distance[s] = 0
            queue.append(s)
    while queue:
        u = queue.popleft()
        for v in adj[u]:
            if distance[v] == -1:
                distance[v] = distance[u] + 1
                queue.append(v)
    return distance

def oranges_rotting(grid):
    if not grid or not grid[0]:
        return 0
    a = [row[:] for row in grid]
    rows, cols = (len(a), len(a[0]))
    queue = deque()
    fresh = 0
    for r in range(rows):
        for c in range(cols):
            if a[r][c] == 2:
                queue.append((r, c, 0))
            if a[r][c] == 1:
                fresh += 1
    time = 0
    while queue:
        r, c, time = queue.popleft()
        for nr, nc in [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]:
            if 0 <= nr < rows and 0 <= nc < cols and (a[nr][nc] == 1):
                a[nr][nc] = 2
                fresh -= 1
                queue.append((nr, nc, time + 1))
    return -1 if fresh else time

def update_matrix(mat):
    if not mat or not mat[0]:
        return []
    rows, cols = (len(mat), len(mat[0]))
    d = [[-1] * cols for _ in range(rows)]
    queue = deque()
    for r in range(rows):
        for c in range(cols):
            if mat[r][c] == 0:
                d[r][c] = 0
                queue.append((r, c))
    while queue:
        r, c = queue.popleft()
        for nr, nc in [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]:
            if 0 <= nr < rows and 0 <= nc < cols and (d[nr][nc] == -1):
                d[nr][nc] = d[r][c] + 1
                queue.append((nr, nc))
    return d

# 示例与回归测试
assert oranges_rotting([[1, 1]]) == -1

assert oranges_rotting([[0, 2]]) == 0

assert oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]) == 4

assert update_matrix([[0, 1], [1, 1]]) == [[0, 1], [1, 2]]

assert update_matrix([[1]]) == [[-1]]

adj = [[1], [0, 2], [1, 3], [2], []]

assert bfs_distances(adj, [0, 3]) == [0, 1, 1, 0, -1]

from random import Random

rng = Random(81)

for _ in range(60):
    n = 7
    adj = [[j for j in range(n) if i != j and rng.randrange(4) == 0] for i in range(n)]
    sources = [0, 3, 5]
    singles = [bfs_distances(adj, [s]) for s in sources]
    ref = [min([d[v] for d in singles if d[v] >= 0], default=-1) for v in range(n)]
    assert bfs_distances(adj, sources) == ref

print('N081: 所有本章断言通过')

N081: 所有本章断言通过


## 11. 代表题与迁移

- **994. Rotting Oranges（canonical）：** 就是本章的 oranges_rotting，练"多源初始化 + 逐分钟分层 + 幸存者返回 -1"。
- **542. 01 Matrix（canonical）：** 就是本章的 update_matrix，练"反向多源（从所有 0 扩散）把 O((RC)²) 降到 O(RC)"的视角转换。
- **1091. Shortest Path in Binary Matrix（canonical）：** 同样的网格 BFS 求最短步数，但它**含对角线共 8 个方向**且要处理起点/终点是 1 的边界，正好检验你有没有把"四方向"当成死记硬背。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。